# Text FTM Attack Runner (Colab + VS Code)

This notebook runs your full experiment on 100 samples with surrogate + black-box evaluation and saves results.

## 1) Prepare Project From GitHub

This step clones your repository into the Colab runtime so GPU execution can access your code and dataset.

Repository used:
- https://github.com/AbdulRehman-gitrep/ftm_text

If your repo is private, replace clone URL with your token-auth URL or use Colab GitHub auth.

In [5]:
# Run this cell first in Colab.
# It clones your GitHub repo into /content and sets FTM_PROJECT_DIR.
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/AbdulRehman-gitrep/ftm_text.git'
CLONE_PARENT = Path('/content')
CLONE_DIR = CLONE_PARENT / 'ftm_text'

if CLONE_DIR.exists():
    print('Repo already exists at', CLONE_DIR)
    # Optional refresh: fetch latest changes
    subprocess.run(['git', '-C', str(CLONE_DIR), 'pull'], check=False)
else:
    CLONE_PARENT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO_URL, str(CLONE_DIR)], check=True)

os.environ['FTM_PROJECT_DIR'] = str(CLONE_DIR)
print('FTM_PROJECT_DIR set to:', os.environ['FTM_PROJECT_DIR'])
print('Contents:', [p.name for p in CLONE_DIR.iterdir()])

FTM_PROJECT_DIR set to: /content/ftm_text
Contents: ['utils', 'requirements.txt', 'data', 'run_attack_colab.ipynb', 'config.py', 'models', '__pycache__', 'main.py', 'attacks', '.git']


In [6]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime

def is_project_dir(p: Path) -> bool:
    return (p / 'main.py').exists() and (p / 'requirements.txt').exists()

def find_project_dir():
    # Optional manual override for any environment.
    env_path = os.environ.get('FTM_PROJECT_DIR')
    if env_path:
        p = Path(env_path).expanduser()
        if is_project_dir(p):
            return p.resolve()

    # Fast checks for common local/Colab layouts.
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/FTM_text/text_ftm_attack'),
        Path('/content/drive/MyDrive/FTM_text/text_ftm_attack'),
    ]
    for p in candidates:
        if is_project_dir(p):
            return p.resolve()

    # Recursive fallback search under common roots.
    search_roots = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]
    for root in search_roots:
        if not root.exists():
            continue
        for main_file in root.rglob('main.py'):
            p = main_file.parent
            if is_project_dir(p):
                return p.resolve()

    return None

PROJECT_DIR = find_project_dir()
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find text_ftm_attack folder. Set FTM_PROJECT_DIR or place project in /content/... and rerun."
    )

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('Python =', sys.executable)

PROJECT_DIR = /content/ftm_text
Python = /usr/bin/python3


## 2) Check GPU

In [7]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    DEVICE = 'cuda:0'
else:
    DEVICE = 'cpu'
print('DEVICE =', DEVICE)

torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4
DEVICE = cuda:0


## 3) Install Dependencies

In [8]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Dependencies installed.')

Dependencies installed.


## 4) Verify Dataset

In [9]:
DATA_CSV = PROJECT_DIR / 'data' / 'IMDB Dataset.csv'
if not DATA_CSV.exists():
    raise FileNotFoundError(f'Missing dataset file: {DATA_CSV}')
print('Dataset found:', DATA_CSV)

Dataset found: /content/ftm_text/data/IMDB Dataset.csv


## 5) Run Attack (100 Samples + Black-Box Eval)

In [ ]:
RUN_TAG = datetime.now().strftime('run_gpu_%Y%m%d_%H%M%S')
SAVE_DIR = PROJECT_DIR / 'results' / RUN_TAG
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = SAVE_DIR / 'run.log'

cmd = [
    sys.executable, 'main.py',
    '--device', DEVICE,
    '--num_samples', '100',
    '--eval',
    '--data_csv', str(DATA_CSV),
    '--save_dir', str(SAVE_DIR),
]

print('Running command:')
print(' '.join(cmd))
print('Full logs will be saved to:', LOG_PATH)

# Print only key progress lines to avoid notebook output truncation.
progress_markers = [
    'Sample ',
    'iter ',
    'Surrogate ASR',
    'Average black-box ASR',
    'Results saved to',
    'DONE',
]

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True,
 )

with open(LOG_PATH, 'w', encoding='utf-8') as log_f:
    for line in proc.stdout:
        log_f.write(line)
        if any(marker in line for marker in progress_markers):
            print(line, end='')

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'Attack run failed with exit code {ret}. Check {LOG_PATH}')

print('Run completed.')
print('SAVE_DIR =', SAVE_DIR)
print('LOG_PATH =', LOG_PATH)

Running command:
/usr/bin/python3 main.py --device cuda:0 --num_samples 100 --eval --data_csv /content/ftm_text/data/IMDB Dataset.csv --save_dir /content/ftm_text/results/run_gpu_20260425_153145
  TEXT FTM ATTACK
Device       : cuda:0
Surrogate    : distilbert-base-uncased-finetuned-sst-2-english
Iterations   : 300
Samples      : 100
Eval on BBMs : True
Loaded 100 samples from /content/ftm_text/data/IMDB Dataset.csv

Loading surrogate model: distilbert-base-uncased-finetuned-sst-2-english

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2177.92it/s, Materializing param=pre_classifier.weight]
  Transformer layers: 6
  Hidden size:        768
  Hooked layers:      [3, 4, 5]

────────────────────────────────────────────────────────────
Sample 1/100
  True label  : 1 (POSITIVE)
  Target label: 0 (NEGATIVE)
  Text: Extremely tense thriller set in the urban chaos of São Paulo, the biggest and ugliest third world ni...
  iter  100/300 | pred=1 target=0 | changed=0.0% | logit[target]=4

In [ ]:
# Optional helper: list result files and persist artifacts.
import glob
import shutil
from pathlib import Path

print('Result directory:', SAVE_DIR)
print('Files:')
for p in sorted(Path(SAVE_DIR).glob('*')):
    print('-', p.name)

zip_path = str(SAVE_DIR) + '.zip'
if not Path(zip_path).exists():
    shutil.make_archive(str(SAVE_DIR), 'zip', root_dir=str(SAVE_DIR))
print('ZIP:', zip_path)

# Try Colab download button.
try:
    from google.colab import files
    print('Use this to download manually if needed:')
    print(f"files.download('{zip_path}')")
except Exception:
    print('Not running in Colab; skip files.download helper.')

# If Google Drive is mounted, copy zip for persistence across runtime restarts.
drive_root = Path('/content/drive/MyDrive')
if drive_root.exists():
    dest_dir = drive_root / 'ftm_results'
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / Path(zip_path).name
    shutil.copy2(zip_path, dest)
    print('Copied zip to Drive:', dest)
else:
    print('Drive not mounted. To persist, mount Drive then copy the zip manually.')

## 6) Load and Print Final Metrics

In [11]:
summary_path = SAVE_DIR / 'results_summary.json'
examples_path = SAVE_DIR / 'examples.txt'

if not summary_path.exists():
    raise FileNotFoundError(f'Summary not found: {summary_path}')

with open(summary_path, 'r', encoding='utf-8') as f:
    summary = json.load(f)

print('===== FINAL METRICS =====')
print('num_samples:', summary.get('num_samples'))
print('surrogate_asr:', summary.get('surrogate_asr'))
print('avg_semantic_similarity:', summary.get('avg_semantic_similarity'))
print('above_sim_threshold:', summary.get('above_sim_threshold'))

bb = summary.get('black_box_asr', {})
if bb:
    print('black_box_asr:')
    for k, v in bb.items():
        print(' -', k, ':', v)
    print('avg_black_box_asr:', summary.get('avg_black_box_asr'))

print('summary_path =', summary_path)
print('examples_path =', examples_path)

===== FINAL METRICS =====
num_samples: 100
surrogate_asr: 5.0
avg_semantic_similarity: 0.9998568391799927
above_sim_threshold: 100
black_box_asr:
 - textattack/bert-base-uncased-imdb : 4.0
 - textattack/roberta-base-imdb : 3.0
 - textattack/xlnet-base-cased-imdb : 2.0
avg_black_box_asr: 3.0
summary_path = /content/ftm_text/results/run_gpu_20260425_153145/results_summary.json
examples_path = /content/ftm_text/results/run_gpu_20260425_153145/examples.txt


## 7) Optional: Zip Results for Download

In [12]:
import shutil
zip_path = str(SAVE_DIR) + '.zip'
shutil.make_archive(str(SAVE_DIR), 'zip', root_dir=str(SAVE_DIR))
print('Created:', zip_path)

Created: /content/ftm_text/results/run_gpu_20260425_153145.zip
